In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input
/kaggle/input/competitions
/kaggle/input/competitions/asl-signs
/kaggle/input/competitions/asl-signs/train_landmark_files
/kaggle/input/competitions/asl-signs/train_landmark_files/36257
/kaggle/input/competitions/asl-signs/train_landmark_files/25571
/kaggle/input/competitions/asl-signs/train_landmark_files/55372
/kaggle/input/competitions/asl-signs/train_landmark_files/26734
/kaggle/input/competitions/asl-signs/train_landmark_files/16069
/kaggle/input/competitions/asl-signs/train_landmark_files/53618
/kaggle/input/competitions/asl-signs/train_landmark_files/27610
/kaggle/input/competitions/asl-signs/train_landmark_files/32319
/kaggle/input/competitions/asl-signs/train_landmark_files/49445
/kaggle/input/competitions/asl-signs/train_landmark_files/28656
/kaggle/input/competitions/asl-signs/train_landmark_files/4718


In [ ]:
import numpy as np # linear algebra
import pandas as pd

df = pd.read_csv('/kaggle/input/competitions/asl-signs/train.csv')

print(df.shape)
df.head()

In [ ]:
df['sign'].value_counts()

In [ ]:
selected_signs = [
    'sleepy',
    'wake',
    'look',
    'who',
    'see',
    'hello',
    'hear',
    'cow',
    'person'
]

filtered_df = df[df['sign'].isin(selected_signs)]

print(filtered_df.shape)
filtered_df.head()

In [ ]:
print(df[df['sign'].isin(selected_signs)]['sign'].value_counts())

In [ ]:
import pandas as pd
import numpy as np

def extract_features(parquet_path):
    pq = pd.read_parquet(parquet_path)

    pq = pq[pq['type'].isin(['left_hand', 'right_hand','pose'])]
    pq[['x', 'y', 'z']] = pq[['x', 'y', 'z']].fillna(0)

    feature_vector = []
    landmark_count={
            'left_hand':21,
            'right_hand':21,
            'pose':33
        }
    for part in ['left_hand', 'right_hand','pose']:

        part_df = pq[pq['type'] == part]
        num_landmarks=landmark_count[part]
       
        for landmark in range(num_landmarks):
        
                lm = part_df[part_df['landmark_index'] == landmark]
    
                for coord in ['x', 'y', 'z']:
    
                    vals = lm[coord].values
    
                    if len(vals) > 0:
                        feature_vector.extend([
                            vals.mean(),
                            vals.std(),
                            vals.min(),
                            vals.max(),
                            vals[0],
                            vals[-1],
                            vals[-1] - vals[0]
                        ])
                    
                    else:
                    
                        feature_vector.extend([0] * 7)

    return np.array(feature_vector)

In [ ]:
sample_path = filtered_df.iloc[0]['path']

full_path = '/kaggle/input/competitions/asl-signs/' + sample_path

vec = extract_features(full_path)

print(vec.shape)

In [ ]:
X = []
y = []

for _, row in filtered_df.iterrows():

    full_path = '/kaggle/input/competitions/asl-signs/' + row['path']

    X.append(extract_features(full_path))
    y.append(row['sign'])

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

In [ ]:
le = LabelEncoder()

y_encoded = le.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print(classification_report(y_test, pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_test, pred)

pd.DataFrame(
    cm,
    index=le.classes_,
    columns=le.classes_
)

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, pred))

In [ ]:
import joblib

joblib.dump(model, "sign_xgb2.pkl")
joblib.dump(le, "label_encoder2.pkl")